# Bacterial Growth Prediction

Machine learning regression project predicting four bacterial growth parameters using experimental input variables. Compares Ridge, Decision Tree, Random Forest and Support Vector Regression.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

### **Introduction**

The provided dataset is part of a regression problem aimed at predicting the growth parameters of bacteria under various experimental conditions. It includes results from experiments involving different bacterial strains and environmental factors, such as CO2 availability, light levels, and sucrose production efficiency. The dataset consists of 9 fields, including 5 input features (n_cyanos, co2, light, SucRatio, Nsample) and 4 target variables (a, mu, tau, a0) representing growth parameters. These parameters—maximum bacteria count (a), growth rate (mu), growth delay time (tau), and initial bacteria level (a0)—serve as the prediction targets. This dataset offers opportunities to explore regression techniques for modeling bacterial growth dynamics.

We'll start by loading the dataset and examining its structure.


---

In [ ]:
# Load the dataset
data = pd.read_csv("fitting-results.csv")

## **1. Exploratory Data Analysis(EDA)**

It performs an initial exploration of the dataset to understand its structure, identify potential issues, and prepare it for further analysis.

**Feature and Target Separation:**
> These are the independent variables or input variables in your dataset.

> They provide the information that the machine learning model uses to make predictions.

**Targets:**
> These are the dependent variables or the outputs the model aims to predict.
> In this dataset, they are  'a', 'mu', 'tau' and 'a0'.

In [ ]:
# Display basic dataset info
print("Dataset Shape:", data.shape) #function provides the number of rows and columns in the dataset, giving a sense of its size.
print("Dataset Info:")
data.info() #displays column data types, memory usage, and non-null values
print("\nSummary Statistics:")
print(data.describe()) #provides key statistics (mean, standard deviation, min, max, etc.) for numerical columns to summarize the dataset's distribution and range.

#Check missing values
print("\nMissing Values:")
print(data.isnull().sum()) #counts the number of missing values in each column, helping to determine if data imputation or cleaning is necessary.

The correlation heatmap provides a visual representation of the pairwise relationships between variables in the dataset. Each cell in the heatmap displays the correlation coefficient, with values ranging from -1 to 1. Positive correlations are represented by warm tones (red), indicating a direct relationship, while negative correlations are shown in cool tones (blue), signifying an inverse relationship.

In [ ]:
# 2. Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

Distribution plots give a view about the underlying pattern and variability of both predictors and target variables in the dataset. Each histogram is overlaid with a kernel density estimate that highlights the shape of the distribution.

In [ ]:
# 3. Distributions of predictors and targets
for col in data.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(data[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

Boxplots are particularly useful for visualizing the spread of data and identifying potential outliers.

In [ ]:
# 4. Boxplots for outliers
for col in data.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col])
    plt.title(f"Boxplot for {col}")
    plt.xlabel(col)
    plt.show()

## 2. Data normalisation

Features are standardised **inside model pipelines**, after the train/validation/test split, so scaling parameters are learned from training folds only.

In [ ]:
features = ['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']
targets = ['a', 'mu', 'tau', 'a0']
X = data[features]
y = data[targets]
print('Input features:', features)
print('Prediction targets:', targets)

## **3. Split the data into train, test and validation sets**

In [ ]:
# 60% training, 20% testing, 20% validation
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_test, X_validate, y_test, y_validate = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print('Train:', X_train.shape, 'Validation:', X_validate.shape, 'Test:', X_test.shape)

## **4. Pipeline**

I implemented four regression pipelines—Ridge Regression, Decision Tree, Random Forest, and Support Vector Machine (SVR)—to evaluate their performance on predicting a continuous target variable

**Ridge Regression Pipeline:-**  This linear model with L2 regularization was chosen to handle datasets with multicollinearity and to prevent overfitting by penalizing large coefficients.
> It is a simple and efficient approach for datasets where the relationship between features and the target is mostly linear.

**Decision Tree Regression Pipeline:-** This model was included to capture non-linear relationships within the data.

> It provides a baseline for non-linear regression tasks.

**Random Forest Regression:-**An ensemble model built from multiple Decision Trees, Random Forest was selected for its ability to generalize well on complex datasets while reducing overfitting.

**Support Vector Machine Regression (SVR):-**  SVR was chosen for its ability to model complex, non-linear relationships by mapping features into higher dimensions using kernels

>It works well for datasets with subtle patterns but requires careful parameter tuning and proper feature scaling.

In [ ]:
# Define regression pipelines
pipelines = {
    "Ridge Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', Ridge())
    ]),
    "Decision Tree": Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', RandomForestRegressor(random_state=42))
    ]),
    "Support Vector Machine": Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', SVR())
    ])
}

Hyperparameter tuning is essential for optimizing model performance and ensuring that the chosen model parameters are best suited to the data.

In [ ]:
# Hyperparameter tuning
param_grids = {
    "Ridge Regression": {'regressor__alpha': [0.1, 1.0, 10.0]},
    "Decision Tree": {'regressor__max_depth': [5, 10, None]},
    "Random Forest": {'regressor__n_estimators': [10, 50, 100]},
    "Support Vector Machine": {'regressor__C': [0.1, 1, 10]}
}

## 5. Model evaluation

Compare tuned regression models using MSE and R² on training and validation data. Then assess the selected model for each target once on the held-out test set.

In [ ]:
# Tune on training data, choose best model on validation data, then test once.
results = {}
best_models = {}
for target in targets:
    print(f"\nEvaluating target: {target}")
    results[target] = {}
    models_for_target = {}
    for name, pipeline in pipelines.items():
        search = GridSearchCV(pipeline, param_grids[name], cv=5, scoring='r2', n_jobs=-1)
        search.fit(X_train, y_train[target])
        best_model = search.best_estimator_
        models_for_target[name] = best_model
        train_pred = best_model.predict(X_train)
        valid_pred = best_model.predict(X_validate)
        results[target][name] = {
            'Best Params': search.best_params_,
            'Train MSE': mean_squared_error(y_train[target], train_pred),
            'Validation MSE': mean_squared_error(y_validate[target], valid_pred),
            'Train R2': r2_score(y_train[target], train_pred),
            'Validation R2': r2_score(y_validate[target], valid_pred),
        }
    winner = max(results[target], key=lambda name: results[target][name]['Validation R2'])
    best_models[target] = (winner, models_for_target[winner])
print('Best models selected using validation data:', {k: v[0] for k,v in best_models.items()})

In [ ]:
# Summary of validation performance for all four targets.
rows = []
for target, scores in results.items():
    for name, metrics in scores.items():
        rows.append({'Target': target, 'Model': name, **metrics})
validation_results = pd.DataFrame(rows)
display(validation_results.sort_values(['Target', 'Validation R2'], ascending=[True, False]))

# Evaluate selected models on held-out test data, only after validation model selection.
test_results = []
for target, (name, model) in best_models.items():
    prediction = model.predict(X_test)
    test_results.append({'Target': target, 'Model': name,
                         'Test MSE': mean_squared_error(y_test[target], prediction),
                         'Test R2': r2_score(y_test[target], prediction)})
display(pd.DataFrame(test_results))

In [ ]:
# Compare model R² on the validation subset for target a.
selected_target = 'a'
model_names = list(results[selected_target])
train_r2 = [results[selected_target][name]['Train R2'] for name in model_names]
valid_r2 = [results[selected_target][name]['Validation R2'] for name in model_names]
x = np.arange(len(model_names))
plt.figure(figsize=(11, 5))
plt.bar(x-0.18, train_r2, 0.36, label='Train R²')
plt.bar(x+0.18, valid_r2, 0.36, label='Validation R²')
plt.xticks(x, model_names, rotation=10)
plt.ylabel('R² Score')
plt.title('Model Comparison for Growth Parameter a')
plt.legend(); plt.tight_layout(); plt.show()

## Conclusion

In the original analysis, Random Forest showed the strongest overall performance across the four prediction targets. This standalone notebook uses scaling inside pipelines and validation-based model selection to avoid preprocessing leakage. **The revised code has not been rerun with the original data**, so its final scores are not yet verified.